# 03 - 보고된 speedup 조건별 검산

**학습 목표**: 논문 Table 2, 4, 6 값을 산술 재계산하고 output length와 concurrency에 따른 경향을 분리합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

모델 benchmark를 다시 실행하는 것이 아니라 공개 수치를 검산합니다.

In [ ]:
# framework별 원시 표시값을 dict로 유지해 서로 다른 실행 환경의 수치를 섞지 않습니다.
frameworks = [
    {'name': 'Transformers', 'ar_latency': 34.850, 'df_latency': 5.474, 'ar_tps': 40.9, 'df_tps': 245.7},
    {'name': 'vLLM', 'ar_latency': 3.032, 'df_latency': 1.408, 'ar_tps': 466.9, 'df_tps': 1002.3},
]
for row in frameworks:
    latency_speedup = row['ar_latency'] / row['df_latency']
    tps_speedup = row['df_tps'] / row['ar_tps']
    print(row['name'], 'latency=', round(latency_speedup, 2), 'TPS=', round(tps_speedup, 2))

length_buckets = [('[0,256]', 1.31), ('(256,512]', 1.55), ('(512,1024]', 1.77),
                  ('(1024,2048]', 2.14), ('(2048,+inf)', 2.30)]
concurrency = [(1, 2.14), (2, 2.11), (4, 2.26), (6, 2.16), (8, 2.12), (16, 1.87), (32, 1.80)]
print('vLLM by output length:', length_buckets)
print('vLLM by concurrency:', concurrency)

assert all(length_buckets[i][1] <= length_buckets[i+1][1] for i in range(len(length_buckets)-1))
assert round(frameworks[0]['ar_latency'] / frameworks[0]['df_latency'], 2) == 6.37
assert round(frameworks[1]['ar_latency'] / frameworks[1]['df_latency'], 2) == 2.15  # 표의 2.14는 더 정밀한 내부 값/반올림 차이


`3.032/1.408`을 표시된 세 자리 값만으로 계산하면 2.153...이므로 2.15입니다. 논문 표의 2.14는 원시 측정값으로 계산한 뒤 각 latency를 반올림했을 가능성이 있습니다. 이런 rounding discrepancy를 억지로 맞추지 않는 것이 재현 검수에 중요합니다.